# NLP feature engineering — sentence embeddings

Sprint 6. Encodes each `review` body into a dense semantic vector with a
pre-trained **SentenceTransformer** (`all-MiniLM-L6-v2`, 384 dims) and
saves the result as `features_embeddings.parquet`, keyed by `wine_id`.

This is the **expensive, compute-once** artifact. Encoding ~135k reviews
takes a few minutes on CPU; everything downstream (PCA reduction, anchor
projection, modelling) is cheap and reads this cache instead of re-encoding.

Design (consistent with `features_basic` / `features_keywords`):
- raw 384-dim vectors stored as **float32** wide columns `emb_000…emb_383`
- keyed by `wine_id` so it joins to the other feature tables 1:1
- no reduction/normalisation baked in — those are modelling choices made later

In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

SILVER_PATH     = r"..\..\.data\wine_reviews_silver.parquet"
EMBEDDINGS_PATH = r"..\..\features\features_embeddings.parquet"
MODEL_NAME      = "all-MiniLM-L6-v2"
EMB_DIM         = 384

df = pd.read_parquet(SILVER_PATH)
print(f"Silver shape: {df.shape}")
print(f"reviews missing: {df['review'].isna().sum():,}  (encoded as empty string)")

c:\Users\Mateusz\PycharmProjects\wines\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Silver shape: (135192, 24)
reviews missing: 17  (encoded as empty string)


## Load model

`all-MiniLM-L6-v2` — 384-dim, fast, strong general-purpose sentence
embeddings. Downloaded from the HuggingFace hub on first run, then cached
locally. Runs on CPU; uses GPU automatically if available.

In [2]:
model = SentenceTransformer(MODEL_NAME)
device = model.device
print(f"Loaded {MODEL_NAME}  |  dim={model.get_sentence_embedding_dimension()}  |  device={device}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1930.54it/s]


Loaded all-MiniLM-L6-v2  |  dim=384  |  device=cpu


C:\Users\Mateusz\AppData\Local\Temp\ipykernel_20088\1978148802.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Loaded {MODEL_NAME}  |  dim={model.get_sentence_embedding_dimension()}  |  device={device}")


## Encode reviews

Batched encoding over the full corpus. `normalize_embeddings=False` keeps
raw vectors (normalise later only if a downstream step needs cosine geometry).
Deterministic — no randomness — so re-running reproduces identical vectors.

In [3]:
reviews = df["review"].fillna("").astype(str).tolist()

embeddings = model.encode(
    reviews,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=False,
).astype(np.float32)

print(f"embeddings: {embeddings.shape}  dtype={embeddings.dtype}")
assert embeddings.shape == (len(df), EMB_DIM)

Batches: 100%|██████████| 2113/2113 [43:35<00:00,  1.24s/it] 


embeddings: (135192, 384)  dtype=float32


## Assemble & save

`wine_id` + `emb_000…emb_383`. One row per wine, joins to `features_basic`
/ `features_keywords` on `wine_id`.

In [4]:
emb_cols = [f"emb_{i:03d}" for i in range(EMB_DIM)]
emb_df = pd.DataFrame(embeddings, columns=emb_cols, index=df.index)
out = pd.concat([df[["wine_id"]], emb_df], axis=1)

out.to_parquet(EMBEDDINGS_PATH, index=False)
size_mb = __import__("os").path.getsize(EMBEDDINGS_PATH) / 1e6
print(f"Saved {out.shape[0]:,} rows x {out.shape[1]} cols ({size_mb:.0f} MB) -> {EMBEDDINGS_PATH}")
out.iloc[:3, :6]

Saved 135,192 rows x 385 cols (315 MB) -> ..\..\.data\features_embeddings.parquet


,wine_id,emb_000,emb_001,emb_002,emb_003,emb_004
0,0,0.043674,-0.002083,0.000393,0.031783,0.011054
1,1,0.025155,-0.071454,0.029646,0.012063,0.038614
2,2,-0.051229,-0.054648,-0.015398,0.007075,0.050622


## Sanity check — semantic neighbours

Quick confirmation the vectors carry meaning: for a sample review, find its
nearest neighbours by cosine similarity. Wines with similar tasting language
should surface (independent of price/region), which is exactly the signal we
hope adds value beyond `features_basic`.

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

# L2-normalise for cosine; take a manageable slice to keep the demo fast
sample_n = 5000
E = embeddings[:sample_n]
E = E / np.linalg.norm(E, axis=1, keepdims=True)

query = 0
sims = cosine_similarity(E[query:query+1], E)[0]
top = np.argsort(-sims)[1:4]  # skip self

print("QUERY:", df["review"].iloc[query][:200], "\n")
for r in top:
    print(f"[{sims[r]:.3f}] {df['review'].iloc[r][:200]}\n")

QUERY: A fruity, inviting nose of dark cherry and baking spice lead the way to a complex, textured and richly layered midpalate of integrated intensity and dried herb in this reserve-worthy wine. Robust and  

[0.843] This rich and relaxed wine is drenched in ripe black-cherry and mellow oak flavors and has plenty of body and rather soft tannins. Baking spices add to the plummy, mulled cherry notes for a sense of d

[0.836] This remarkably impressive wine opens in an intriguingly edge, earthy nose, leading the way to a lengthy and complex experience on the palate. Balanced red-fruit, spice and forest notes accent supple,

[0.832] This is an effusively delicious and impressive wine, offering warmth and depth around rich flavors of baked cherry and strawberry. With a lightness of baking spice and herb it shows good integration o



## Next steps (downstream, not here)

- **Predictive**: PCA-reduce 384 → ~32–64 dims, add to the model matrix in
  `models/01_models.ipynb`, compare R² vs the 0.6404 baseline.
- **Interpretable**: anchor-projection — cosine of each review vs flavour
  anchor sentences → `emb_fruity`, `emb_tannic`, … (good for the report/BI).
- Keep this notebook as the single source of the raw embedding cache.